In [12]:
from summary_chatbot.api.chain import State, call_model, summarize_history, should_continue, print_update, summarise
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage

In [13]:
with open('../data/story.txt', 'r') as file:
    contenu = file.read()
summary = summarise(contenu)
print(summary)

Daniel took a walk in the forest behind his town to clear his head after a long and exhausting day. The peaceful surroundings helped him to relax and appreciate the beauty of nature. Although the walk did not solve his problems, it reminded him of the bigger picture and gave him a sense of calm. He knew that he could always return to the quiet path whenever he needed a break from his busy life.


In [14]:
# Define a new graph
workflow = StateGraph(State)

# Define the conversation node and the summarize node
workflow.add_node("conversation", lambda input: call_model(state=input, conversation_summary=summary))
workflow.add_node(summarize_history)

# Set the entrypoint as conversation
workflow.add_edge(START, "conversation")

# We now add a conditional edge
workflow.add_conditional_edges(
    # First, we define the start node. We use `conversation`.
    # This means these are the edges taken after the `conversation` node is called.
    "conversation",
    # Next, we pass in the function that will determine which node is called next.
    should_continue,
)

# We now add a normal edge from `summarize_conversation` to END.
# This means that after `summarize_conversation` is called, we end.
workflow.add_edge("summarize_history", END)

# Finally, we compile it!
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)


In [ ]:
config = {"configurable": {"thread_id": "4"}}
input_message = HumanMessage(content="Quels sont les personnages principaux de FF7 ?")
input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

input_message = HumanMessage(content="Parmi eux qui à une épée ?")
input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

input_message = HumanMessage(content="Quel est le nom en français ?")
input_message.pretty_print()
for event in app.stream({"messages": [input_message]}, config, stream_mode="updates"):
    print_update(event)

================================ Human Message =================================

Quels sont les personnages principaux de FF7 ?
================================== Ai Message ==================================

Les personnages principaux de Final Fantasy VII sont Cloud Strife, Tifa Lockhart, Aerith Gainsborough, Barret Wallace, Red XIII, Cid Highwind, et Yuffie Kisaragi. Ces personnages sont au cœur de l'histoire et jouent des rôles importants tout au long du jeu.
================================ Human Message =================================

Parmi eux qui à une épée ?
================================== Ai Message ==================================

Cloud Strife est le personnage principal de Final Fantasy VII qui est connu pour utiliser une épée massive appelée "Buster Sword". C'est son arme emblématique et il la manie avec une grande habileté lors des combats.
================================ Human Message =================================

Quel est le nom en français ?
===========